In [1]:
"""
Decentralized RS — Train/Test Split Ratio Experiment (ML-100K)
==============================================================
Compares three split strategies:  90/10  |  80/20  |  70/30
Val set is always 20% of the training portion (proportional).
Metrics tracked per ratio:
  • Test RMSE
  • Convergence speed (epochs to best val loss)
  • Communication cost (total commute × parameter bytes)

Drop in project root alongside src/ and dataset/.
Run:  python split_ratio_experiment.py
"""

from pathlib import Path
import os

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")



Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [3]:
import copy, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.optim import SGD
from tqdm import tqdm

from src.models.MatrixFactorization import UMF
from src.graphs import create_scale_free_graph
from src.users import User
from src.data_utils import create_dataloader
from src.training.decentralized import (
    decentralized_train_n_epochs,
    decentralized_validate_loop,
)

In [4]:
warnings.filterwarnings("ignore")
torch.manual_seed(0)
np.random.seed(42)


# ──────────────────────────────────────────────────────────────────────────────
# Hyper-parameters  (mirrors your notebook exactly)
# ──────────────────────────────────────────────────────────────────────────────
HP = dict(
    n_factors    = 30,
    sparse       = False,
    batch_size   = 10,
    lr = 0.043245636749499355,
    weight_decay = 0.24293301237845355,
    mom = 0.6590721600407826,
    graph_seed   = 1,
    n_epochs     = 50,
    loader_type  = "rs",
    # DP-SGD
    use_dp       = True,
    dp_clip_norm = 1.0,
    dp_noise_std = 0.01,
)



# Val is always 20 % of the training portion (proportional)
VAL_FRAC = 0.20


# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────
def make_train_test_split(full_df: pd.DataFrame, train_frac: float):
    """Split full interaction data into train / test by train_frac."""
    return train_test_split(full_df, train_size=train_frac, random_state=42)


def make_val_split(train_df: pd.DataFrame, val_frac: float = VAL_FRAC):
    """Carve val out of train proportionally."""
    return train_test_split(train_df, test_size=val_frac, random_state=0)


def build_users(n_users: int, n_items: int, hp: dict) -> dict:
    users = {}
    for i in tqdm(range(n_users), desc="  Init users", leave=False):
        model = UMF(n_items, n_factors=hp["n_factors"], sparse=hp["sparse"])
        opt   = SGD(model.parameters(), lr=hp["lr"],
                    weight_decay=hp["weight_decay"], momentum=hp["mom"])
        users[i] = User(id=i, model=model, optimizer=opt, model_name="umf")
    return users


def dp_epsilon(sigma, n_steps, n_train, batch_size, delta=1e-5):
    q = batch_size / n_train
    return np.sqrt(2 * n_steps * np.log(1 / delta)) * q / sigma


# ──────────────────────────────────────────────────────────────────────────────
# One experiment
# ──────────────────────────────────────────────────────────────────────────────
def run_experiment(label: str, train_df: pd.DataFrame,
                   val_df: pd.DataFrame, test_df: pd.DataFrame,
                   n_items: int, hp: dict) -> dict:

    print(f"\n{'─'*60}")
    print(f"  Ratio {label}  |  train={len(train_df)}  val={len(val_df)}"
          f"  test={len(test_df)}")
    print(f"{'─'*60}")

    n_users = train_df["user_id"].nunique()

    train_loader = create_dataloader(df=train_df, dl_type=hp["loader_type"],
                                     batch_size=hp["batch_size"])
    val_loader   = create_dataloader(df=val_df,  dl_type=hp["loader_type"])
    test_loader  = create_dataloader(df=test_df, dl_type=hp["loader_type"])

    users = build_users(n_users, n_items, hp)
    graph = create_scale_free_graph(n_users=n_users,  seed=hp["graph_seed"])

    torch.manual_seed(0)
    t0 = time.time()
    train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=hp["n_epochs"],
        graph=graph,
    )
    elapsed = time.time() - t0

    test_rmse         = float(decentralized_validate_loop(users, test_loader))
    best_val          = float(min(val_losses))
    best_epoch        = int(np.argmin(val_losses)) + 1   # 1-indexed
    epochs_run        = len(train_losses)

    # Communication cost: commute × n_factors × 4 bytes (float32)
    param_bytes        = hp["n_factors"] * 4
    total_commute      = int(sum(commutes['commute']))
    comm_cost_mb       = round(total_commute * param_bytes / 1e6, 3)
    avg_commute_epoch  = round(total_commute / max(epochs_run, 1), 1)

    # Privacy budget at current noise level
    eps = dp_epsilon(hp["dp_noise_std"], epochs_run * len(train_loader),
                     len(train_df), hp["batch_size"])

    result = dict(
        label             = label,
        n_train           = len(train_df),
        n_val             = len(val_df),
        n_test            = len(test_df),
        n_users           = n_users,
        n_items           = n_items,
        test_rmse         = round(test_rmse, 6),
        best_val_loss     = round(best_val, 6),
        best_epoch        = best_epoch,
        epochs_run        = epochs_run,
        train_losses      = [round(x, 6) for x in train_losses],
        val_losses        = [round(x, 6) for x in val_losses],
        time_per_epoch    = [round(x, 3) for x in time_per_epoch],
        commutes          = commutes,
        total_commute     = total_commute,
        comm_cost_mb      = comm_cost_mb,
        avg_commute_epoch = avg_commute_epoch,
        elapsed_s         = round(elapsed, 2),
        dp_epsilon        = round(eps, 4),
        dp_noise_std      = hp["dp_noise_std"],
    )

    print(f"  ✓  Test RMSE: {test_rmse:.4f}  |  Best Val @ epoch {best_epoch}"
          f"  |  Comm: {comm_cost_mb} MB  |  ε={eps:.2f}  |  {elapsed:.1f}s")
    return result




In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# Leave-Out-N Experiment: hold out a fixed number of ratings per user (node)
# Sequence: N_LEAVE_OUT in [5, 10, 15, 20]
#
# Strategy per user:
#   - Users with >= N ratings : last N ratings (by row order) → test
#                                remaining rows (after val carve) → train
#   - Users with < N ratings  : skipped entirely (excluded from all splits)
#
# Val set: 20% of each user's training rows, sampled randomly (proportional).
# ID re-indexing is applied AFTER filtering so embeddings stay contiguous.
# ──────────────────────────────────────────────────────────────────────────────

# ── Load ratings ─────────────────────────────────────────────────────────────
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
#ratings = pd.read_csv("ratings.csv")
ratings = pd.read_csv('dataset/u.data',
                       sep='\t', names=column_names).iloc[:, 0:3]

print(f"Raw ratings: {len(ratings):,}  |  "
      f"Users: {ratings['user_id'].nunique()}  |  "
      f"Items: {ratings['item_id'].nunique()}")

# ── Leave-out-N split per user ────────────────────────────────────────────────
N_LEAVE_OUT_SEQ = [5, 10, 15, 20]   # sequence to benchmark
VAL_FRAC        = 0.20               # fraction of per-user train rows for val


def leave_out_n_split(ratings_df: pd.DataFrame,
                      n_leave: int,
                      val_frac: float = VAL_FRAC,
                      random_state: int = 0):
    """
    Per-user leave-out-N split.

    For each user with >= n_leave ratings:
      - Test  : last n_leave rows (preserves natural ordering).
      - Train : remaining rows, minus val_frac held out for validation.
      - Val   : val_frac of the per-user train rows.

    Users with < n_leave ratings are skipped entirely.

    Returns
    -------
    train_df, val_df, test_df  : DataFrames with columns
                                  [user_id, item_id, rating]
                                  IDs re-indexed to contiguous integers.
    meta : dict with split statistics.
    """
    rng = np.random.default_rng(random_state)

    train_rows, val_rows, test_rows = [], [], []
    skipped_users = 0

    for uid, grp in ratings_df.groupby("user_id"):
        grp = grp.reset_index(drop=True)

        # Skip users with too few ratings
        if len(grp) < n_leave:
            skipped_users += 1
            continue

        # Last n_leave rows → test
        test_idx  = grp.index[-n_leave:]
        train_pool = grp.drop(index=test_idx)

        if len(train_pool) == 0:
            skipped_users += 1
            continue

        # Random val split from training pool
        n_val = max(1, int(np.floor(len(train_pool) * val_frac)))
        val_idx_local = rng.choice(len(train_pool), size=n_val, replace=False)
        val_mask = np.zeros(len(train_pool), dtype=bool)
        val_mask[val_idx_local] = True

        val_rows.append(train_pool.iloc[val_mask])
        train_rows.append(train_pool.iloc[~val_mask])
        test_rows.append(grp.iloc[test_idx])

    train_df = pd.concat(train_rows, ignore_index=True)
    val_df   = pd.concat(val_rows,   ignore_index=True)
    test_df  = pd.concat(test_rows,  ignore_index=True)

    # Only keep items seen in training (cold-start filter)
    train_items = set(train_df.item_id.unique())
    val_df  = val_df[val_df.item_id.isin(train_items)].reset_index(drop=True)
    test_df = test_df[test_df.item_id.isin(train_items)].reset_index(drop=True)

    # Re-index user/item IDs to contiguous integers
    all_users  = np.unique(pd.concat([train_df, val_df, test_df]).user_id.values)
    all_items  = np.unique(pd.concat([train_df, val_df, test_df]).item_id.values)
    u2i = {u: i for i, u in enumerate(all_users)}
    a2i = {a: i for i, a in enumerate(all_items)}

    for df in [train_df, val_df, test_df]:
        df["user_id"] = df["user_id"].map(u2i)
        df["item_id"] = df["item_id"].map(a2i)

    meta = dict(
        n_leave      = n_leave,
        n_users      = len(all_users),
        n_items      = len(all_items),
        skipped_users= skipped_users,
        n_train      = len(train_df),
        n_val        = len(val_df),
        n_test       = len(test_df),
    )
    return train_df, val_df, test_df, meta


# ── Run experiment for each leave-out-N value ─────────────────────────────────
all_results = []

for n_leave in N_LEAVE_OUT_SEQ:
    print(f"\n{'═'*60}")
    print(f"  Leave-Out-N = {n_leave}")
    print(f"{'═'*60}")

    train_df, val_df, test_df, meta = leave_out_n_split(
        ratings, n_leave=n_leave, val_frac=VAL_FRAC
    )

    print(f"  Users: {meta['n_users']}  (skipped: {meta['skipped_users']})  "
          f"|  Items: {meta['n_items']}")
    print(f"  Train: {meta['n_train']:,}  |  "
          f"Val: {meta['n_val']:,}  |  "
          f"Test: {meta['n_test']:,}  "
          f"(= {meta['n_users']} users × {n_leave} ratings each)")

    res = run_experiment(
        label    = f"leave_{n_leave}",
        train_df = train_df,
        val_df   = val_df,
        test_df  = test_df,
        n_items  = meta["n_items"],
        hp       = HP,
    )
    res["n_leave"]       = n_leave
    res["skipped_users"] = meta["skipped_users"]
    all_results.append(res)


Raw ratings: 100,000  |  Users: 943  |  Items: 1682

════════════════════════════════════════════════════════════
  Leave-Out-N = 5
════════════════════════════════════════════════════════════
  Users: 943  (skipped: 0)  |  Items: 1650
  Train: 76,595  |  Val: 18,653  |  Test: 4,712  (= 943 users × 5 ratings each)

────────────────────────────────────────────────────────────
  Ratio leave_5  |  train=76595  val=18653  test=4712
────────────────────────────────────────────────────────────


  0%|          | 0/76595 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.5367 | Validation Loss: 1.0292 | Time Elapsed: 31.355587 sec |Commute: 105095 | Commute 
Cost: 3917834250

  0%|          | 0/76595 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.3235 | Validation Loss: 0.9891 | Time Elapsed: 34.733496 sec |Commute: 105095 | Commute 
Cost: 3917834250

...

  ✓  Test RMSE: 1.0076  |  Best Val @ epoch 13  |  Comm: 168.274 MB  |  ε=66.53  |  431.5s

════════════════════════════════════════════════════════════
  Leave-Out-N = 15
════════════════════════════════════════════════════════════
  Users: 943  (skipped: 0)  |  Items: 1645
  Train: 69,051  |  Val: 16,767  |  Test: 14,129  (= 943 users × 15 ratings each)

────────────────────────────────────────────────────────────
  Ratio leave_15  |  train=69051  val=16767  test=14129
────────────────────────────────────────────────────────────


  0%|          | 0/69051 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.7429 | Validation Loss: 1.0248 | Time Elapsed: 28.361951 sec |Commute: 95231 | Commute 
Cost: 3521255745

  0%|          | 0/69051 [00:00<?, ?it/s]

ValueError: 

In [8]:
# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "═"*80)
print(f"{'Ratio':<8} {'TrainN':>7} {'TestN':>7} {'TestRMSE':>10}"
      f" {'BestEpoch':>10} {'CommMB':>9} {'ε':>7}")
print("═"*80)
for r in all_results:
    print(f"{r['label']:<8} {r['n_train']:>7} {r['n_test']:>7}"
          f" {r['test_rmse']:>10.4f} {r['best_epoch']:>10}"
          f" {r['comm_cost_mb']:>9.2f} {r['dp_epsilon']:>7.2f}")
print("═"*80)

out = Path("split_ratio_results.json")
out.write_text(json.dumps(all_results, indent=2))
print(f"\nResults saved → {out}")


════════════════════════════════════════════════════════════════════════════════
Ratio     TrainN   TestN   TestRMSE  BestEpoch    CommMB       ε
════════════════════════════════════════════════════════════════════════════════
leave_5    76595    4712     0.9956         11    151.34   60.06
leave_10   72823    9423     1.0076         13    168.27   66.53
════════════════════════════════════════════════════════════════════════════════


TypeError: Object of type float32 is not JSON serializable